In [ ]:
import numpy as np
import pandas as pd
from rfdetr import RFDETRNano
import supervision as sv
import json
import os
import shutil 
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from collections import defaultdict
import glob
import matplotlib.pyplot as plt
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision, Precision, Recall
from PIL import Image
from zipfile import ZipFile
import matplotlib.pyplot as plt 

2025-12-07 13:35:44.603696: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765114544.969181     136 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765114545.097862     136 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
model = RFDETRNano()

rf-detr-nano.pth: 100%|██████████| 349M/349M [00:03<00:00, 116MiB/s] 


Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


In [3]:
path= os.getcwd()
path

'/kaggle/working'

In [ ]:

# ==============================================================================
# HILFSFUNKTIONEN
# ==============================================================================

def filter_coco_json(original_coco_data: dict, image_ids_to_keep: np.ndarray, output_path: str):
    """Erstellt eine neue COCO JSON-Datei, die nur die angegebenen Bild-IDs enthält."""
    
    new_data = {
        'images': [],
        'annotations': [],
        'categories': original_coco_data.get('categories', []),
        'info': original_coco_data.get('info', {}),
        'licenses': original_coco_data.get('licenses', [])
    }
    
    ids_set = set(image_ids_to_keep)
    
    # 1. Bilder filtern
    kept_image_ids = set()
    for img in original_coco_data['images']:
        if img['id'] in ids_set:
            new_data['images'].append(img)
            kept_image_ids.add(img['id'])
    
    # 2. Annotationen filtern
    for ann in original_coco_data['annotations']:
        if ann['image_id'] in kept_image_ids:
            new_data['annotations'].append(ann)
            
    with open(output_path, 'w') as f:
        json.dump(new_data, f)
        
    return output_path

# ==============================================================================
# 0. KONFIGURATION & INITIALISIERUNG
# ==============================================================================
N_SPLITS = 5
TEST_SIZE = 0.2
RANDOM_SEED = 42
EPOCHS = 100

# --- ANPASSEN SIE DIESE VARIABLEN AN IHRE UMGEBUNG ---
MASTER_COCO_JSON_PATH = r"/kaggle/input/archive-zip/train/_annotations.coco.json" 
BASE_DATA_PATH = r"/kaggle/input/archive-zip/train" # TATSÄCHLICHER BILDER-ROOT
TEMP_DIR = os.path.abspath("./cv_temp_coco/") 
# CLASS_NAMES = ['DEFECT'] 
# --------------------------------------------------------

# Initialisierung
os.makedirs(TEMP_DIR, exist_ok=True)
all_fold_metrics = defaultdict(list)
ABSOLUTE_PATH = os.path.join(BASE_DATA_PATH, '_annotations.coco.json')
TRAIN_POOL_JSON_PATH = os.path.join(TEMP_DIR, 'train_pool.json')
FINAL_TEST_JSON_PATH = os.path.join(TEMP_DIR, 'test_final.json')

# NON STRATIFIED

In [ ]:
# ==============================================================================
# 0. VORBEREITUNG: NUR IMAGE-IDS LADEN (KEINE LABELS NÖTIG)
# ==============================================================================

# 1. Daten laden
with open(ABSOLUTE_PATH, 'r') as f:
    train_pool_data = json.load(f)

# 2. Mapping: Image ID -> Dateiname
image_id_to_filename = {img['id']: img['file_name'] for img in train_pool_data['images']}

# 3. X (Image IDs) erstellen
print("Erstelle Index für K-Fold...")
X = np.array([img['id'] for img in train_pool_data['images']])

print(f"Anzahl Bilder im Pool: {len(X)}")

# ==============================================================================
# 1. K-FOLD CV LOOP (STANDARD K-FOLD)
# ==============================================================================

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_results = []

# Kopier-Helfer
def copy_images_to_fold(id_list, dest_dir):
    for image_id in id_list:
        file_name = image_id_to_filename[image_id]
        source_path = os.path.join(BASE_DATA_PATH, file_name)
        dest_path = os.path.join(dest_dir, file_name)
        try:
            shutil.copyfile(source_path, dest_path)
        except Exception as e:
            print(f"Fehler bei {file_name}: {e}")

# Hilfsfunktion für Evaluation
def evaluate_subset(dir_path, json_filename, model_weights_path):
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=dir_path,
        annotations_path=os.path.join(dir_path, json_filename)
    )
    
    inference_model = RFDETRNano(pretrain_weights=model_weights_path)
    
    targets = []
    predictions = []
    
    for path, image, annotations in dataset:
        pil_image = Image.open(path).convert("RGB")
        detections = inference_model.predict(pil_image, threshold=0) 
        
        targets.append(annotations)
        predictions.append(detections)
    
    map_metric = MeanAveragePrecision()
    prec_metric = Precision()
    rec_metric = Recall()
    
    map_result = map_metric.update(predictions, targets).compute()
    prec_result = prec_metric.update(predictions, targets).compute()
    rec_result = rec_metric.update(predictions, targets).compute()


    cm_metric = sv.ConfusionMatrix.from_detections(
    predictions=predictions,
    targets=targets,
    classes=dataset.classes
    )
    
    return map_result, prec_result, rec_result, cm_metric

for fold, (train_val_index, test_index) in enumerate(kf.split(X)):
    print(f"\n==================================================")
    print(f"--- FOLD {fold+1}/{N_SPLITS} ---")
    print(f"==================================================")

    # 1. Splits definieren (NUR basierend auf Index, ohne Labels)
    
    # Test-Teil (wird NICHT trainiert, nur evaluiert)
    X_test_ids = X[test_index]
    
    # Train+Val Teil
    X_train_val_ids = X[train_val_index]

    # Nested Split: Train vs Val
    # Hier KEIN 'stratify' Argument mehr!
    X_train_ids, X_val_ids = train_test_split(
        X_train_val_ids, 
        test_size=0.15, 
        random_state=RANDOM_SEED
    )

    print(f"  Train: {len(X_train_ids)} | Val: {len(X_val_ids)} | Test: {len(X_test_ids)}")

    # 2. Verzeichnisse vorbereiten
    train_dir = os.path.join(TEMP_DIR, 'train')
    val_dir   = os.path.join(TEMP_DIR, 'valid')
    test_dir  = os.path.join(TEMP_DIR, 'test')

    for d in [train_dir, val_dir, test_dir]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    # 3. JSONs filtern
    filter_coco_json(train_pool_data, X_train_ids, os.path.join(train_dir, '_annotations.coco.json'))
    filter_coco_json(train_pool_data, X_val_ids,   os.path.join(val_dir, '_annotations.coco.json'))
    filter_coco_json(train_pool_data, X_test_ids,  os.path.join(test_dir, '_annotations.coco.json'))

    # 4. Bilder kopieren
    print("  Kopiere Bilder...")
    copy_images_to_fold(X_train_ids, train_dir)
    copy_images_to_fold(X_val_ids, val_dir)
    copy_images_to_fold(X_test_ids, test_dir)

    # 5. Training
    run_name = f'rf_detr_fold_{fold+1}'
    project_dir = './rf-detr_runs'
    
    model.train(
        dataset_dir=TEMP_DIR, 
        epochs=EPOCHS,
        name=run_name,
        project=project_dir,
        val=True,
        run_test=False 
    )

    # Pfad zu den Gewichten
    weights_path = "/kaggle/working/output/checkpoint_best_total.pth"

      # -----------------------------------------------------------
    # 6. A) EVALUATION AUF DEM VALIDATION-SET (Trainings-Check)
    # -----------------------------------------------------------
    print(f"-> Starte Inference auf VALIDATION-Set Fold {fold+1}...")
    val_map, val_prec, val_rec, conf_mat_val = evaluate_subset(val_dir, '_annotations.coco.json', weights_path)
    print(f"   [VAL] mAP50-95: {val_map} | Precision: {val_prec} | Recall: {val_rec}")

    try:
        conf_mat_val.plot()
        plt.title(f"Validation Confusion Matrix - Fold {fold+1}")
        plt.show() 
        plt.close() # Speicher freigeben
    except Exception as e:
        print(f"Fehler beim Plotten der Val-Matrix: {e}")

    
    # -----------------------------------------------------------
    # 6. B) EVALUATION AUF DEM TEST-SET (Unbekannte Daten)
    # -----------------------------------------------------------
    print(f"-> Starte Inference auf TEST-Set Fold {fold+1}...")
    test_map, test_prec, test_rec, conf_mat_test  = evaluate_subset(test_dir, '_annotations.coco.json', weights_path)
    print(f"   [TEST] mAP50-95: {test_map} | Precision: {test_prec} | Recall: {test_rec}")


    try:
        conf_mat_test.plot()
        plt.title(f"Test Confusion Matrix - Fold {fold+1}")
        plt.show()
        plt.close()
    except Exception as e:
        print(f"Fehler beim Plotten der Test-Matrix: {e}")
    
    # 7. Ergebnisse speichern
    fold_res = {
        'Fold': fold + 1,
        # Validation Metriken
        'Val_mAP50': val_map.map50,
        'Val_mAP50-95': val_map.map50_95,
        'Val_Precision': val_prec.precision_at_50,
        'Val_Recall': val_rec.recall_at_50,
        # Test Metriken
        'Test_mAP50': test_map.map50,
        'Test_mAP50-95': test_map.map50_95,
        'Test_Precision': test_prec.precision_at_50,
        'Test_Recall': test_rec.recall_at_50
    }
    fold_results.append(fold_res)

    # 8. Aufräumen
    shutil.rmtree(train_dir)
    shutil.rmtree(val_dir)
    shutil.rmtree(test_dir)

# ==============================================================================
# 2. FINALER BERICHT
# ==============================================================================
df = pd.DataFrame(fold_results)

# Spalten ordnen
cols = ['Fold', 
        'Val_mAP50', 'Val_mAP50-95', 'Val_Precision', 'Val_Recall', 
        'Test_mAP50', 'Test_mAP50-95', 'Test_Precision', 'Test_Recall']
df = df[cols]

print("\n##################################################")
print("## ERGEBNISSE: VALIDIERUNG vs. TEST ##")
print("##################################################")
df.to_csv("ergebnisse.csv")

print(df)

# STRATIFIED

In [ ]:
# ==============================================================================
# 0. VORBEREITUNG: IMAGE-IDS UND LABELS (y) FÜR STRATIFIZIERUNG LADEN
# ==============================================================================

# 1. Daten laden
with open(ABSOLUTE_PATH, 'r') as f:
    train_pool_data = json.load(f)

# 2. Mapping: Image ID -> Dateiname
image_id_to_filename = {img['id']: img['file_name'] for img in train_pool_data['images']}

# 3. X (Image IDs) und y (Labels) erstellen
print("Extrahiere Labels für Stratified K-Fold...")
X_ids = []
y_labels = []

# Helper: Annotationen nach Image-ID gruppieren
anns_by_image = {}
for ann in train_pool_data['annotations']:
    img_id = ann['image_id']
    if img_id not in anns_by_image: anns_by_image[img_id] = []
    anns_by_image[img_id].append(ann['category_id'])

# Iteriere durch alle Bilder für X und y
for img in train_pool_data['images']:
    img_id = img['id']
    X_ids.append(img_id)
    
    cats = anns_by_image.get(img_id, [])
    if cats:
        # STRATEGIE: Nimm die erste Kategorie für die Stratifizierung.
        # Falls du eine "Defect"-Priorisierung brauchst (Klasse 0 ist wichtiger als 1),
        # kannst du das hier anpassen.
        y_labels.append(cats[0]) 
    else:
        y_labels.append(-1) # -1 für Hintergrund / Keine Annotation

X = np.array(X_ids)
y = np.array(y_labels) # <--- Unser y für StratifiedKFold

print(f"Anzahl Bilder im Pool: {len(X)}")
from collections import Counter
print(f"Klassenverteilung: {Counter(y)}")


# ==============================================================================
# 1. STRATIFIED K-FOLD CV LOOP
# ==============================================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_results = []

def copy_images_to_fold(id_list, dest_dir):
    for image_id in id_list:
        file_name = image_id_to_filename[image_id]
        source_path = os.path.join(BASE_DATA_PATH, file_name)
        dest_path = os.path.join(dest_dir, file_name)
        try:
            shutil.copyfile(source_path, dest_path)
        except Exception as e:
            print(f"Fehler bei {file_name}: {e}")

# Hilfsfunktion für Evaluation
def evaluate_subset(dir_path, json_filename, model_weights_path):
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=dir_path,
        annotations_path=os.path.join(dir_path, json_filename)
    )
    
    inference_model = RFDETRNano(pretrain_weights=model_weights_path)
    
    targets = []
    predictions = []
    
    for path, image, annotations in dataset:
        pil_image = Image.open(path).convert("RGB")
        detections = inference_model.predict(pil_image, threshold=0) 
        
        targets.append(annotations)
        predictions.append(detections)
    
    map_metric = MeanAveragePrecision()
    prec_metric = Precision()
    rec_metric = Recall()
    
    map_result = map_metric.update(predictions, targets).compute()
    prec_result = prec_metric.update(predictions, targets).compute()
    rec_result = rec_metric.update(predictions, targets).compute()


    cm_metric = sv.ConfusionMatrix.from_detections(
    predictions=predictions,
    targets=targets,
    classes=dataset.classes
    )
    
    return map_result, prec_result, rec_result, cm_metric


# HIER: split(X, y) statt split(X)
for fold, (train_val_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\n==================================================")
    print(f"--- FOLD {fold+1}/{N_SPLITS} (Stratified) ---")
    print(f"==================================================")

    # 1. Splits
    X_test_ids = X[test_index]
    y_test     = y[test_index] # (Wird hier nicht weiter verwendet, aber ist Teil des Splits)
    
    X_train_val_ids = X[train_val_index]
    y_train_val     = y[train_val_index] # Wichtig für den nächsten stratify-Split

    # Nested Split: Train vs Val (ebenfalls stratifiziert!)
    X_train_ids, X_val_ids = train_test_split(
        X_train_val_ids, 
        test_size=0.15, 
        stratify=y_train_val, # <--- Stratify aktiv
        random_state=RANDOM_SEED
    )

    print(f"  Train: {len(X_train_ids)} | Val: {len(X_val_ids)} | Test: {len(X_test_ids)}")

    # 2. Verzeichnisse vorbereiten
    train_dir = os.path.join(TEMP_DIR, 'train')
    val_dir   = os.path.join(TEMP_DIR, 'valid')
    test_dir  = os.path.join(TEMP_DIR, 'test')

    for d in [train_dir, val_dir, test_dir]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    # 3. JSONs filtern
    filter_coco_json(train_pool_data, X_train_ids, os.path.join(train_dir, '_annotations.coco.json'))
    filter_coco_json(train_pool_data, X_val_ids,   os.path.join(val_dir, '_annotations.coco.json'))
    filter_coco_json(train_pool_data, X_test_ids,  os.path.join(test_dir, '_annotations.coco.json'))

    # 4. Bilder kopieren
    print("  Kopiere Bilder...")
    copy_images_to_fold(X_train_ids, train_dir)
    copy_images_to_fold(X_val_ids, val_dir)
    copy_images_to_fold(X_test_ids, test_dir)

    # 5. Training
    run_name = f'rf_detr_fold_{fold+1}'
    project_dir = './rf-detr_runs'
    
    model.train(
        dataset_dir=TEMP_DIR, 
        epochs=EPOCHS,
        name=run_name,
        project=project_dir,
        val=True,
        run_test=False 
    )

    # Pfad zu den Gewichten
    weights_path = "/kaggle/working/output/checkpoint_best_total.pth"

      # -----------------------------------------------------------
    # 6. A) EVALUATION AUF DEM VALIDATION-SET (Trainings-Check)
    # -----------------------------------------------------------
    print(f"-> Starte Inference auf VALIDATION-Set Fold {fold+1}...")
    val_map, val_prec, val_rec, conf_mat_val = evaluate_subset(val_dir, '_annotations.coco.json', weights_path)
    print(f"   [VAL] mAP50-95: {val_map} | Precision: {val_prec} | Recall: {val_rec}")

    try:
        conf_mat_val.plot()
        plt.title(f"Validation Confusion Matrix - Fold {fold+1}")
        plt.show() 
        plt.close() # Speicher freigeben
    except Exception as e:
        print(f"Fehler beim Plotten der Val-Matrix: {e}")

    
    # -----------------------------------------------------------
    # 6. B) EVALUATION AUF DEM TEST-SET (Unbekannte Daten)
    # -----------------------------------------------------------
    print(f"-> Starte Inference auf TEST-Set Fold {fold+1}...")
    test_map, test_prec, test_rec, conf_mat_test  = evaluate_subset(test_dir, '_annotations.coco.json', weights_path)
    print(f"   [TEST] mAP50-95: {test_map} | Precision: {test_prec} | Recall: {test_rec}")


    try:
        conf_mat_test.plot()
        plt.title(f"Test Confusion Matrix - Fold {fold+1}")
        plt.show()
        plt.close()
    except Exception as e:
        print(f"Fehler beim Plotten der Test-Matrix: {e}")
    
    # 7. Ergebnisse speichern
    fold_res = {
        'Fold': fold + 1,
        # Validation Metriken
        'Val_mAP50': val_map.map50,
        'Val_mAP50-95': val_map.map50_95,
        'Val_Precision': val_prec.precision_at_50,
        'Val_Recall': val_rec.recall_at_50,
        # Test Metriken
        'Test_mAP50': test_map.map50,
        'Test_mAP50-95': test_map.map50_95,
        'Test_Precision': test_prec.precision_at_50,
        'Test_Recall': test_rec.recall_at_50
    }
    fold_results.append(fold_res)

    # 8. Aufräumen
    shutil.rmtree(train_dir)
    shutil.rmtree(val_dir)
    shutil.rmtree(test_dir)

# ==============================================================================
# 2. FINALER BERICHT
# ==============================================================================
df = pd.DataFrame(fold_results)

# Spalten ordnen
cols = ['Fold', 
        'Val_mAP50', 'Val_mAP50-95', 'Val_Precision', 'Val_Recall', 
        'Test_mAP50', 'Test_mAP50-95', 'Test_Precision', 'Test_Recall']
df = df[cols]

print("\n##################################################")
print("## ERGEBNISSE: VALIDIERUNG vs. TEST ##")
print("##################################################")
df.to_csv("ergebnisse.csv")

print(df)